# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2 dataset: Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as specified below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will enumerate the record sets, each field, and available columns, referencing them **by their `@id`** for subsequent extraction.

In [ ]:
# Get all available record sets and their ids
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"    - {field['@id']}")
                else:
                    print(f"    - {field}")
        if 'column' in rs:
            print("  Columns:")
            for col in rs['column']:
                if isinstance(col, dict):
                    print(f"    - {col['@id']}")
                else:
                    print(f"    - {col}")


## 3. Data Extraction
Here, we load actual data from available record sets into pandas DataFrames for analysis.

You should select the `@id` of the desired record set and its relevant fields from the previous overview step.
We'll attempt to extract from all available record sets.

In [ ]:
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet '@id': {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for RecordSet '@id': {record_set_id}. Reason: {e}")

# Preview the first table if available
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"Preview of records from RecordSet '@id': {first_id}")
    display(dataframes[first_id].head())
else:
    print("No dataframes could be loaded from dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as basic filtering, normalization, and grouping.

All fields are referenced using their `@id`. Select a numeric field (`@id`) for numeric analysis (e.g., a coefficient or log-likelihood column). Adjust the `numeric_field_id` and `group_field_id` as found in Section 2 or 3.

In [ ]:
# Example: Choose the first loaded record set and identify numeric fields by inspecting the DataFrame
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Use first record set
    df = dataframes[record_set_id]
    print(f'Columns for RecordSet {record_set_id}:')
    print(list(df.columns))
    
    # Guess a numeric field
    sample_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64', 'float32', 'int32']]
    if sample_numeric_fields:
        numeric_field_id = sample_numeric_fields[0]  # Use the first numeric column
    else:
        # Fall back: Try to convert all columns to numeric and pick one
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='raise')
                numeric_field_id = col
                df[col] = converted
                break
            except Exception:
                continue
        else:
            print('No numeric field found in this DataFrame.')
            numeric_field_id = None

    if numeric_field_id:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical variable
        group_field_candidates = [col for col in df.columns if 'ward' in col.lower() or 'county' in col.lower() or df[col].dtype == 'object']
        group_field_id = None
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
                print(grouped_df.head())
    else:
        print("No numeric field available for EDA in this record set.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields.

We attempt a histogram of the selected numeric field and, where possible, a grouped bar plot for categorical breakdown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we found a numeric field in the EDA section
if dataframes and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- In this notebook, we demonstrated how to load, explore, process, and visualize a Croissant dataset in Python using the `mlcroissant` library, referencing all entities by their `@id` fields.
- All record sets, fields, and columns were handled dynamically according to the dataset metadata.
- Further analysis should be guided by the dataset content and research questions—update field selections and visualizations for deeper or domain-specific insights.